In [0]:
%sql
CREATE TABLE main.default.market_1m_candles (
trade_time TIMESTAMP,
symbol STRING,
open DOUBLE,
high DOUBLE,
low DOUBLE,
close DOUBLE,
volume BIGINT
)
USING DELTA;

In [0]:
%sql
CREATE TABLE main.default.samco_1m_candles_RELIANCE (
trade_time TIMESTAMP,
symbol STRING,
open DOUBLE,
high DOUBLE,
low DOUBLE,
close DOUBLE,
volume BIGINT
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.text("/Volumes/main/default/vol1/")

In [0]:
from pyspark.sql.functions import regexp_replace, col

clean_df = df.withColumn(
    "clean_json",
    regexp_replace(col("value"), r'\\\"', '"')
)

clean_df = clean_df.withColumn(
    "clean_json",
    regexp_replace(col("clean_json"), r'^"|"$', '')
)

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import *

schema = StructType([
    StructField("response", StructType([
        StructField("data", StructType([
            StructField("symbol", StringType()),
            StructField("ltt", StringType()),
            StructField("taq", StringType()),
            StructField("tbq", StringType()),
            StructField("bidValues", ArrayType(
                StructType([
                    StructField("price", StringType()),
                    StructField("qty", StringType()),
                    StructField("no", StringType())
                ])
            ))
        ]))
    ]))
])

parsed = clean_df.withColumn(
    "json",
    from_json(col("clean_json"), schema)
)

In [0]:
result = parsed.select(
    col("json.response.data.symbol").alias("symbol"),
    col("json.response.data.ltt").alias("trade_time"),
    col("json.response.data.taq").cast("long").alias("volume"),
    col("json.response.data.bidValues")[0]["price"].cast("double").alias("close")
)

result.show()

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/main/default/vol2/2885_NSE_schemaLocation")
    .load("/Volumes/main/default/vol2/2885_NSE/")
)

In [0]:
from pyspark.sql.functions import *

In [0]:
parsed_df = df.select(
    col("response.data.sym").alias("symbol"),
    col("response.data.o").cast("double").alias("open"),
    col("response.data.h").cast("double").alias("high"),
    col("response.data.l").cast("double").alias("low"),
    col("response.data.ltp").cast("double").alias("close"),
    col("response.data.vol").cast("long").alias("volume"),
    # col("response.data.ltt").alias("trade_time"),
    to_timestamp(col("response.data.ltt"),"dd MMM yyyy, hh:mm:ss a").alias("trade_time")
)

In [0]:
grouped_1m_df = parsed_df.withWatermark("trade_time", "2 seconds").groupBy(window(col("trade_time"), "1 minute"), col("symbol")).agg(
    first("close").alias("open"),
max("close").alias("high"),
min("close").alias("low"),
last("close").alias("close"),
sum("volume").alias("volume")
).select(to_timestamp("window.end").alias("trade_time"), "symbol", "open", "high", "low", "close", "volume")

In [0]:
query = (
    parsed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/main/default/vol1_checkpoint_vol")
    .trigger(processingTime="5 seconds")
    .toTable("main.default.market_1m_candles")
)

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", "/Volumes/main/default/vol2/2885_NSE_schemaLocation")
    .load("/Volumes/main/default/vol2/2885_NSE/")
)

In [0]:
# from pyspark.sql.functions import regexp_replace, col

clean_df = df.withColumn(
    "clean_json",
    regexp_replace(col("value"), r'\\\"', '"')
)

clean_df = clean_df.withColumn(
    "clean_json",
    regexp_replace(col("clean_json"), r'^"|"$', '')
)

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import *

schema = StructType([
    StructField("response", StructType([
        StructField("data", StructType([
            StructField("symbol", StringType()),
            StructField("ltt", StringType()),
            StructField("taq", StringType()),
            StructField("tbq", StringType()),
            StructField("bidValues", ArrayType(
                StructType([
                    StructField("price", StringType()),
                    StructField("qty", StringType()),
                    StructField("no", StringType())
                ])
            ))
        ]))
    ]))
])

parsed = clean_df.withColumn(
    "json",
    from_json(col("clean_json"), schema)
)

In [0]:
result = parsed.select(
    col("json.response.data.symbol").alias("symbol"),
    col("json.response.data.ltt").alias("trade_time"),
    col("json.response.data.taq").cast("long").alias("volume"),
    col("json.response.data.bidValues")[0]["price"].cast("double").alias("close")
).withColumn("trade_time", to_timestamp(col("trade_time"), "dd/MM/yyyy HH:mm:ss"))

In [0]:
grouped_1m_df = result.withWatermark("trade_time", "2 seconds").groupBy(window(col("trade_time"), "1 minute"), col("symbol")).agg(
    first("close").alias("open"),
max("close").alias("high"),
min("close").alias("low"),
last("close").alias("close"),
sum("volume").alias("volume")
).select(to_timestamp("window.end").alias("trade_time"), "symbol", "open", "high", "low", "close", "volume")

In [0]:
query = (
    grouped_1m_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/main/default/vol2/2885_NSE_checkpoint")
    .trigger(processingTime="5 seconds")
    .toTable("main.default.samco_1m_candles_RELIANCE")
)